# Analisa Klasifikasi Berita

## Mount Google Drive & Baca File Berita

In [ ]:
# Install library yang diperlukan (jalankan sekali saja)
!pip install Sastrawi scikit-learn gensim nltk

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Baca file CSV
import pandas as pd

file_path = "/content/drive/MyDrive/PPW/Berita_UTS.csv"  # sesuaikan path kamu
df = pd.read_csv(file_path, encoding='utf-8')

print("✅ Total entri:", len(df))
print("📄 Kolom tersedia:", df.columns)
df.head()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Total entri: 1500
📄 Kolom tersedia: Index(['No', 'judul', 'berita', 'tanggal', 'kategori', 'link'], dtype='object')


,No,judul,berita,tanggal,kategori,link
0,1,Airlangga Harap Kenaikan UMP Tingkatkan Daya B...,Menteri Koordinator (Menko) Bidang Perekonomia...,"Minggu, 01 Des 2024 23:40 WIB",Ekonomi,https://www.cnnindonesia.com/ekonomi/202412012...
1,2,PT SIER Beri Penghargaan untuk 50 Tenant Terba...,"Dalam rangka memeriahkan hari jadi ke-50, PT S...","Minggu, 01 Des 2024 20:45 WIB",Ekonomi,https://www.cnnindonesia.com/ekonomi/202412012...
2,3,Prabowo Bakal Bentuk Kementerian Penerimaan Ne...,Wacana Presiden Prabowo Subianto akan membentu...,"Minggu, 01 Des 2024 19:40 WIB",Ekonomi,https://www.cnnindonesia.com/ekonomi/202412011...
3,4,Sinergi Kemenag & BPJS Ketenagakerjaan Lindung...,BPJS Ketenagakerjaan dan Kementerian Agama (Ke...,"Minggu, 01 Des 2024 19:03 WIB",Ekonomi,https://www.cnnindonesia.com/ekonomi/202412011...
4,5,Pemerintah Segera Bentuk Satgas PHK Usai Tetap...,Pemerintah akan segera membentuk Satuan Tugas ...,"Minggu, 01 Des 2024 19:00 WIB",Ekonomi,https://www.cnnindonesia.com/ekonomi/202412011...


## Preprocessing (Bersihkan & Tokenisasi Teks)

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

In [ ]:
# --- Tahap 2: Preprocessing ---
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('indonesian'))

# Baca file CSV
file_path = "/content/drive/MyDrive/PPW/Berita_UTS.csv"  # updated file path
df = pd.read_csv(file_path, encoding='utf-8') # Added encoding based on cell 9dxd_che8aPw


# Gunakan kolom "berita" (bukan isi_berita)
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

df['clean_text'] = df['berita'].apply(clean_text)
print(df[['berita', 'clean_text']].head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


                                              berita  \
0  Menteri Koordinator (Menko) Bidang Perekonomia...   
1  Dalam rangka memeriahkan hari jadi ke-50, PT S...   
2  Wacana Presiden Prabowo Subianto akan membentu...   
3  BPJS Ketenagakerjaan dan Kementerian Agama (Ke...   
4  Pemerintah akan segera membentuk Satuan Tugas ...   

                                          clean_text  
0  menteri koordinator menko bidang perekonomian ...  
1  rangka memeriahkan pt surabaya industrial esta...  
2  wacana presiden prabowo subianto membentuk mem...  
3  bpjs ketenagakerjaan kementerian agama kemenag...  
4  pemerintah membentuk satuan tugas pemutusan hu...  


## Ekstraksi Fitur dengan LDA (Topic Modeling)

In [ ]:
# === 2. Buat stopwords Bahasa Indonesia ===
factory = StopWordRemoverFactory()
stop_words_indonesia = factory.get_stop_words()

# === 3. Vectorizer & LDA ===
vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words=stop_words_indonesia)
X = vectorizer.fit_transform(df['clean_text'])

n_topics = 4  # karena ada 4 kategori berita
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
lda_matrix = lda.fit_transform(X)

# === 4. Proporsi topik per dokumen ===
topic_dist_df = pd.DataFrame(lda_matrix, columns=[f"Topik {i+1}" for i in range(n_topics)])
topic_dist_df.insert(0, "Dokumen", range(1, len(df)+1))
topic_dist_df["Dominan Topik"] = topic_dist_df.iloc[:, 1:].idxmax(axis=1)

print("\n--- Tabel: Proporsi Topik per Dokumen ---")
print(topic_dist_df.head(15))

# === 5. Proporsi kata per topik ===
terms = vectorizer.get_feature_names_out()
topic_words_list = []

for topic_idx, topic in enumerate(lda.components_):
    top_words_idx = topic.argsort()[::-1]  # urutkan dari besar ke kecil
    for i in range(len(top_words_idx)):    # tampilkan semua kata di setiap topik
        topic_words_list.append({
            "Topik": f"Topik {topic_idx+1}",
            "Kata": terms[top_words_idx[i]],
            "Proporsi": topic[top_words_idx[i]] / topic.sum()
        })

topic_words_df = pd.DataFrame(topic_words_list)

print("\n--- Tabel: Proporsi Kata dalam Topik ---")
print(topic_words_df.head(30))  # tampilkan 30 kata pertama


--- Tabel: Proporsi Topik per Dokumen ---
    Dokumen   Topik 1   Topik 2   Topik 3   Topik 4 Dominan Topik
0         1  0.203516  0.000837  0.794796  0.000851       Topik 3
1         2  0.997664  0.000766  0.000794  0.000776       Topik 1
2         3  0.001033  0.015738  0.882381  0.100848       Topik 3
3         4  0.440597  0.180075  0.377990  0.001338       Topik 1
4         5  0.263262  0.001045  0.734618  0.001076       Topik 3
5         6  0.072653  0.167615  0.431988  0.327743       Topik 3
6         7  0.063714  0.001078  0.934108  0.001101       Topik 3
7         8  0.166977  0.000974  0.690490  0.141558       Topik 3
8         9  0.754264  0.001473  0.019730  0.224533       Topik 1
9        10  0.198129  0.001366  0.367174  0.433331       Topik 4
10       11  0.731536  0.000534  0.267376  0.000554       Topik 1
11       12  0.792101  0.000801  0.206253  0.000845       Topik 1
12       13  0.115384  0.001135  0.781084  0.102397       Topik 3
13       14  0.373843  0.001903  

## Encode Label & Split Data

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Encode kategori berita ke angka
le = LabelEncoder()
y = le.fit_transform(df['kategori'])

# Bagi data train & test
X_train, X_test, y_train, y_test = train_test_split(
    topic_features,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("✅ Split data selesai")
print("Data train:", X_train.shape, "Data test:", X_test.shape)

✅ Split data selesai
Data train: (1200, 5) Data test: (300, 5)


## Klasifikasi dengan Naïve Bayes & SVM

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Naïve Bayes
nb = GaussianNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

print("=== 📊 Naive Bayes ===")
print("Akurasi:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb, target_names=le.classes_))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_nb))

# SVM
svm = SVC(kernel='linear', probability=True)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

print("\n=== 🤖 SVM ===")
print("Akurasi:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm, target_names=le.classes_))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))

=== 📊 Naive Bayes ===
Akurasi: 0.81
               precision    recall  f1-score   support

      Ekonomi       0.76      0.73      0.75        75
Internasional       0.77      0.85      0.81        75
     Nasional       0.70      0.65      0.68        75
     Olahraga       1.00      1.00      1.00        75

     accuracy                           0.81       300
    macro avg       0.81      0.81      0.81       300
 weighted avg       0.81      0.81      0.81       300

Confusion Matrix:
 [[55 10 10  0]
 [ 0 64 11  0]
 [17  9 49  0]
 [ 0  0  0 75]]

=== 🤖 SVM ===
Akurasi: 0.82
               precision    recall  f1-score   support

      Ekonomi       0.76      0.73      0.75        75
Internasional       0.76      0.91      0.83        75
     Nasional       0.75      0.64      0.69        75
     Olahraga       1.00      1.00      1.00        75

     accuracy                           0.82       300
    macro avg       0.82      0.82      0.82       300
 weighted avg       0.82 

# Clustering Dokumen Spam

## Mount Google Drive & Baca File Berita

In [ ]:
# Install library yang diperlukan (jalankan sekali saja)
!pip install Sastrawi scikit-learn gensim nltk

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Baca file CSV
import pandas as pd

file_path = "/content/drive/MyDrive/PPW/spam.csv"  # sesuaikan path kamu
df = pd.read_csv(file_path, encoding='latin-1')  # latin-1 sering dipakai utk dataset spam
print("✅ Total entri:", len(df))
print("📄 Kolom tersedia:", df.columns)
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Total entri: 5572
📄 Kolom tersedia: Index(['id', 'Text', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], dtype='object')


,id,Text,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,1,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,2,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,3,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,4,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,5,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [ ]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)  #
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['clean_text'] = df['Text'].apply(clean_text) # Use 'Text' column

print("🧼 Contoh hasil preprocessing:")
print(df[['Text', 'clean_text']].head())

🧼 Contoh hasil preprocessing:
                                                Text  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   

                                          clean_text  
0  go until jurong point crazy available only in ...  
1                            ok lar joking wif u oni  
2  free entry in a wkly comp to win fa cup final ...  
3        u dun say so early hor u c already then say  
4  nah i don t think he goes to usf he lives arou...  


## Tokenisasi dan Stopword

In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    return " ".join(tokens)

df['clean_text'] = df['clean_text'].apply(remove_stopwords)
print("🧠 Contoh hasil stopword removal:")
print(df[['clean_text']].head())

🧠 Contoh hasil stopword removal:
                                          clean_text
0  go jurong point crazy available bugis n great ...
1                            ok lar joking wif u oni
2  free entry wkly comp win fa cup final tkts st ...
3                u dun say early hor u c already say
4             nah think goes usf lives around though


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Ekstrasi Fitur dengan TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=1000)  # ambil 1000 fitur teratas
X = vectorizer.fit_transform(df['clean_text'])

print("✅ TF-IDF selesai")
print("🔸 Bentuk fitur:", X.shape)

✅ TF-IDF selesai
🔸 Bentuk fitur: (5572, 1000)


## Clustering dengan K-Means

In [ ]:
from sklearn.cluster import KMeans

n_clusters = 2  # misalnya kita cluster jadi 2 (spam vs ham)
model = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df['cluster'] = model.fit_predict(X)

print("✅ Clustering selesai")
df[['clean_text', 'cluster']].head(10)

✅ Clustering selesai


,clean_text,cluster
0,go jurong point crazy available bugis n great ...,0
1,ok lar joking wif u oni,0
2,free entry wkly comp win fa cup final tkts st ...,0
3,u dun say early hor u c already say,0
4,nah think goes usf lives around though,0
5,freemsg hey darling week word back like fun st...,0
6,even brother like speak treat like aids patent,0
7,per request melle melle oru minnaminunginte nu...,0
8,winner valued network customer selected receiv...,1
9,mobile months u r entitled update latest colou...,1


## Analisis Hasil Cluster

In [ ]:
# Hitung distribusi cluster
print("Distribusi cluster:")
print(df['cluster'].value_counts())

# Lihat contoh teks dari setiap cluster
for c in range(n_clusters):
    print(f"\n📂 Contoh dari Cluster {c}:")
    print(df[df['cluster'] == c]['clean_text'].head(5).tolist())

Distribusi cluster:
cluster
0    5053
1     519
Name: count, dtype: int64

📂 Contoh dari Cluster 0:
['go jurong point crazy available bugis n great world la e buffet cine got amore wat', 'ok lar joking wif u oni', 'free entry wkly comp win fa cup final tkts st may text fa receive entry question std txt rate c apply', 'u dun say early hor u c already say', 'nah think goes usf lives around though']

📂 Contoh dari Cluster 1:
['winner valued network customer selected receivea prize reward claim call claim code kl valid hours', 'mobile months u r entitled update latest colour mobiles camera free call mobile update co free', 'urgent week free membership prize jackpot txt word claim c www dbuk net lccltd pobox ldnw rw', 'rodger burns msg tried call reply sms free nokia mobile free camcorder please call delivery tomorrow', 'congrats year special cinema pass call c suprman v matrix starwars etc free bx ip pm dont miss']


## Menyimpan Hasil Clustering

In [ ]:
output_path = "/content/drive/MyDrive/PPW/hasil_clustering_spam.csv"
df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"💾 Hasil clustering disimpan ke: {output_path}")

💾 Hasil clustering disimpan ke: /content/drive/MyDrive/PPW/hasil_clustering_spam.csv
